In [1]:
# MCP SCENARIO: “Smart HR Onboarding Assistant”
# 🧩 Scenario Background
# You are working in a company called XYZ Corp.
# New employees often face challenges during onboarding, such as:
# - Trouble accessing payroll portal
# - Confusion about leave policies
# - Difficulty setting up email accounts
# - Questions about training schedules
# 👉 Instead of emailing HR or waiting for responses, employees use an AI Onboarding Bot.

# 🤖 What this Bot Should Do
# When a new hire types a question/problem:
# - Understand the query (e.g., “I can’t log into payroll”)
# - Decide if escalation to HR is needed
# - Identify:
# - Category (Payroll / Policy / IT Setup / Training)
# - Priority (High / Medium)
# - Create a support ticket if required
# - Provide instant guidance (FAQs, step-by-step instructions)
# - Show confirmation and next steps

# 🧠 How MCP Fits Here
# This way, the MCP framework is reused in a Human Resources context, where the AI assistant
# streamlines onboarding, reduces HR workload, and ensures employees feel supported from day one.
# Would you like me to design another variation in a customer service setting (like retail or banking),
# so you can compare how MCP adapts across industries?


!pip install groq

# ==============================
# IMPORTS & API KEY
# ==============================
import json
from google.colab import userdata
from groq import Groq

# Load API key securely
api_key = userdata.get("GROQ_API_KEY")
client = Groq(api_key=api_key)

# STEP 0: DATABASE
tickets_db = []

# STEP 1: TOOL (Ticket System)
def create_ticket(issue, priority, category):
    ticket_id = f"HR{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket

# STEP 2: MODEL (Groq LLM)
def analyze_input(user_input):

    prompt = f"""
    You are an HR assistant.

    Extract:
    - category (Payroll / Policy / IT Setup / Training / General)
    - priority (High / Medium)

    Respond ONLY in JSON format like:
    {{
        "category": "...",
        "priority": "..."
    }}

    Query: {user_input}
    """

    try:
        res = client.chat.completions.create(
            model="llama-3.1-70b-versatile",  # ✅ FIXED
            messages=[{"role": "user", "content": prompt}]
        )

        content = res.choices[0].message.content.strip()

        # Debug print (optional)
        print("🔍 LLM Raw Output:", content)

        data = json.loads(content)
        return data["category"].lower(), data["priority"].lower()

    except Exception as e:
        print("⚠️ Error in analysis:", e)
        return "general", "medium"

# STEP 3: DECISION ENGINE (MCP)
def should_call_tool(user_input):

    prompt = f"""
    Does this query require creating a support ticket?

    Answer ONLY "yes" or "no".

    Query: {user_input}
    """

    try:
        res = client.chat.completions.create(
            model="llama-3.1-8b-instant",  # ✅ FIXED
            messages=[{"role": "user", "content": prompt}]
        )

        decision = res.choices[0].message.content.lower().strip()

        print("🧠 Decision LLM:", decision)

        return "yes" in decision

    except Exception as e:
        print("⚠️ Error in decision:", e)
        return True  # fallback

# STEP 4: MCP AGENT
def mcp_agent(user_input):

    print("\n🧠 Input:", user_input)

    if should_call_tool(user_input):

        print("➡️ Tool call required")

        category, priority = analyze_input(user_input)

        print(f"📊 Category: {category}, Priority: {priority}")

        result = create_ticket(user_input, priority, category)

        return f"""
        ✅ Ticket Created Successfully!

        🎫 Ticket ID: {result['ticket_id']}
        📂 Category: {result['category']}
        ⚡ Priority: {result['priority']}

        HR team will contact you soon.
        """

    else:
        print("➡️ AI response (no ticket)")

        try:
            res = client.chat.completions.create(
                model="llama-3.1-8b-instant",  # ✅ FIXED
                messages=[{"role": "user", "content": user_input}]
            )

            return "🤖 " + res.choices[0].message.content

        except Exception as e:
            return f"⚠️ Error generating response: {e}"

# STEP 5: RUN LOOP
print("🚀 Smart HR MCP Assistant (Groq) Started\nType 'exit' to stop\n")

while True:
    query = input("Enter your query: ")

    if query.lower() == "exit":
        print("👋 Exiting...")
        break

    response = mcp_agent(query)
    print(response)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.5 MB/s eta 0:00:00
🚀 Smart HR MCP Assistant (Groq) Started
Type 'exit' to stop

Enter your query: I can't access payroll

🧠 Input: I can't access payroll
🧠 Decision LLM: yes
➡️ Tool call required
⚠️ Error in analysis: Error code: 400 - {'error': {'message': 'The model `llama-3.1-70b-versatile` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
📊 Category: general, Priority: medium

        ✅ Ticket Created Successfully!

        🎫 Ticket ID: HR1000
        📂 Category: general
        ⚡ Priority: medium

        HR team will contact you soon.
        
Enter your query: What is the leave policy?

🧠 Input: What is the leave policy?
🧠 Decision LLM: no
➡️ AI response (no ticket)
🤖 The leave policy varies depending on the country, company, or organization